# 147 — Model Merging (SLERP)

**What you'll learn:**
- What SLERP (Spherical Linear Interpolation) is and why it's better than simple averaging for weight tensors
- How to merge two model `state_dict`s at a blending coefficient `t`
- What the `t` parameter controls: `t=0.0` is model A, `t=1.0` is model B, `t=0.5` is a 50/50 blend
- How to compare outputs from model A, model B, and the merged model

---
**Source:** `examples/147-model-merging/`  
**Models:** `HuggingFaceTB/SmolLM2-135M-Instruct` and `SmolLM2-360M-Instruct`  
**Part A** is fully CPU-safe with tiny tensors. **Part B** requires enough RAM to hold two model copies.

In [ ]:
%pip install -q transformers torch

---
## Part A — CPU-Safe Concept Demonstrations

All cells below run on CPU with tiny tensors — no model downloads required.

### A1 — Why SLERP Instead of Linear Interpolation?

#### Linear interpolation (LERP)

```
LERP(t, v0, v1) = (1 - t) * v0 + t * v1
```

Simple, but has a problem: LERP travels through the *interior* of the sphere, **shrinking the magnitude** of the blended vector.

#### Spherical linear interpolation (SLERP)

```
SLERP(t, v0, v1) = sin((1-t)ω)/sin(ω) * v0  +  sin(t*ω)/sin(ω) * v1

where ω = arccos(v0 · v1 / (|v0| |v1|))  ← angle between the two vectors
```

SLERP travels **along the surface** of the unit sphere, preserving magnitude at every step.

#### Why does this matter for model weights?

Neural network weight matrices live in high-dimensional space. Their magnitude encodes how strongly a layer responds. LERP shrinks this magnitude near `t=0.5`, effectively *damping* the model.  
SLERP preserves the activation scale throughout the interpolation — the merged model behaves more like a true blend of the originals.

```
                     v1 (model B)
                   /
  SLERP arc →    *   ← t=0.5 blend (on sphere surface, full magnitude)
               /
             v0 (model A)

vs LERP:     + ← midpoint (inside sphere, reduced magnitude)
```

In [ ]:
# A2 — Demo slerp() from tools.py with tiny CPU tensors
import torch

def slerp(t: float, v0: torch.Tensor, v1: torch.Tensor) -> torch.Tensor:
    """Exact copy of tools.slerp() — spherical linear interpolation."""
    v0_f = v0.float()
    v1_f = v1.float()
    dot = (v0_f * v1_f).sum() / (v0_f.norm() * v1_f.norm() + 1e-8)
    dot = dot.clamp(-1, 1)
    omega = torch.acos(dot)
    if omega.abs() < 1e-4:  # nearly parallel — fall back to LERP
        return ((1 - t) * v0_f + t * v1_f).to(v0.dtype)
    so = torch.sin(omega)
    return (torch.sin((1 - t) * omega) / so * v0_f + torch.sin(t * omega) / so * v1_f).to(v0.dtype)


# Two simple 4-element weight vectors
v0 = torch.tensor([1.0, 0.0, 0.0, 0.0])  # "model A" weights
v1 = torch.tensor([0.0, 1.0, 0.0, 0.0])  # "model B" weights (perpendicular to A)

print("Interpolating between v0 and v1 at different t values:")
print(f"  v0 (model A)   = {v0.tolist()}  magnitude = {v0.norm():.4f}")
print(f"  v1 (model B)   = {v1.tolist()}  magnitude = {v1.norm():.4f}")
print()

print(f"{'t':<6} {'SLERP result':<40} {'SLERP magnitude':<18} {'LERP magnitude'}")
print("-" * 78)

for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    slerp_result = slerp(t, v0, v1)
    lerp_result  = (1 - t) * v0 + t * v1

    vals = [f"{x:.3f}" for x in slerp_result.tolist()]
    slerp_mag = slerp_result.norm().item()
    lerp_mag  = lerp_result.norm().item()

    print(f"{t:<6} {str(vals):<40} {slerp_mag:<18.4f} {lerp_mag:.4f}")

print()
print("Key observation: SLERP magnitude stays at ~1.0 throughout.")
print("LERP magnitude dips to ~0.71 at t=0.5 (sqrt(2)/2 for perpendicular vectors).")

In [ ]:
# A3 — Demo merge_state_dicts() on a minimal mock state_dict
import torch

def merge_state_dicts(sd_a: dict, sd_b: dict, t: float) -> dict:
    """Exact copy of tools.merge_state_dicts()."""
    def _slerp(t, v0, v1):
        v0_f, v1_f = v0.float(), v1.float()
        dot = (v0_f * v1_f).sum() / (v0_f.norm() * v1_f.norm() + 1e-8)
        dot = dot.clamp(-1, 1)
        omega = torch.acos(dot)
        if omega.abs() < 1e-4:
            return ((1 - t) * v0_f + t * v1_f).to(v0.dtype)
        so = torch.sin(omega)
        return (torch.sin((1 - t) * omega) / so * v0_f + torch.sin(t * omega) / so * v1_f).to(v0.dtype)

    merged = {}
    for key in sd_a:
        if (key in sd_b
                and sd_a[key].shape == sd_b[key].shape
                and sd_a[key].is_floating_point()):
            merged[key] = _slerp(t, sd_a[key], sd_b[key])
        else:
            merged[key] = sd_a[key]  # keep model A's value for non-matching keys
    return merged


torch.manual_seed(0)

# Mock state dicts for two tiny "models"
sd_a = {
    "layer.weight": torch.randn(3, 3),
    "layer.bias":   torch.randn(3),
    "embed.weight": torch.randint(0, 10, (5, 4)).float(),  # integer-like (kept from A)
}
sd_b = {
    "layer.weight": torch.randn(3, 3),
    "layer.bias":   torch.randn(3),
    "embed.weight": torch.randint(0, 10, (5, 4)).float(),
}

merged = merge_state_dicts(sd_a, sd_b, t=0.5)

print("Merging two mock state_dicts at t=0.5\n")

for key in sd_a:
    a_val = sd_a[key].flatten()[:3].tolist()
    b_val = sd_b[key].flatten()[:3].tolist()
    m_val = merged[key].flatten()[:3].tolist()
    print(f"  {key}")
    print(f"    model A (first 3): {[round(x,3) for x in a_val]}")
    print(f"    model B (first 3): {[round(x,3) for x in b_val]}")
    print(f"    merged  (first 3): {[round(x,3) for x in m_val]}")
    print()

print("Merged values are spherically interpolated between A and B.")

### A4 — The `t` Parameter: What Each Value Means

The SLERP coefficient `t` is the only knob you turn when merging models.

| `t` value | Result | Description |
|-----------|--------|-------------|
| `0.0` | Pure model A | Identical to model A — SLERP returns v0 |
| `0.1` | Mostly A, hint of B | 10% B influence; A's personality dominates |
| `0.3` | A-leaning blend | Good if A is the primary model and B adds a skill |
| `0.5` | Equal blend | 50/50 mix; outputs are a genuine blend of both |
| `0.7` | B-leaning blend | Good if B is the primary model |
| `0.9` | Mostly B, hint of A | 90% B influence |
| `1.0` | Pure model B | Identical to model B — SLERP returns v1 |

#### Practical merging tips

- Merging works best when both models **share the same architecture** (same layer shapes).
- Models fine-tuned from the same base (e.g. both fine-tuned from LLaMA-3) merge more coherently than unrelated models.
- `t=0.5` is the most common value. Try `t=0.3` or `t=0.7` to tilt toward one model's strengths.
- If shapes differ (e.g. embedding dimensions), `merge_state_dicts()` keeps model A's weights for that layer.

---
## Part B — Full Model Merge

> **RAM REQUIRED (no GPU needed)**  
> This loads two copies of SmolLM2 (135M + 360M parameters) into RAM simultaneously.  
> Minimum: **~2 GB free RAM** (the models are float32 on CPU).  
> Note: because the two SmolLM2 models have **different sizes**, mismatched layers are kept from model A.  
> For a cleaner merge, use two models of identical size fine-tuned from the same checkpoint.

In [ ]:
# B1 — Full SLERP merge (from workflow.py)
import sys
sys.path.insert(0, '/content')  # Colab path; adjust if running locally

from src.workflow import create_workflow

workflow = create_workflow()

state = {
    "model_a": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "model_b": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "t": 0.5,
    "outputs_a": [],
    "outputs_b": [],
    "outputs_merged": [],
}

print(f"Merging: {state['model_a']}")
print(f"   with: {state['model_b']}")
print(f"SLERP t={state['t']} (0.0=A only, 0.5=equal blend, 1.0=B only)\n")

result = workflow(state)

EVAL_PROMPTS = [
    "What is the capital of France?",
    "Explain gradient descent in one sentence.",
    "Write a haiku about Python.",
]

for i, prompt in enumerate(EVAL_PROMPTS):
    print(f"Prompt: {prompt}")
    print(f"  Model A : {result['outputs_a'][i][:100]}")
    print(f"  Model B : {result['outputs_b'][i][:100]}")
    print(f"  Merged  : {result['outputs_merged'][i][:100]}")
    print()